In [1]:
!uv pip install -q grad-cam

In [2]:
import os
import random
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Callable, Dict, Tuple
import logging

import numpy as np
import pandas as pd
import cv2

import torch
from torch import nn, optim
import torch.backends.cudnn as cudnn
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet34, ResNet34_Weights


from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from itertools import cycle

from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2
import traceback # Added traceback import

from tqdm.notebook import tqdm  # Better progress bars for notebooks

#import torch.nn as nn

In [3]:
# mount drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Change the current working directory
new_dir = "/content/drive/MyDrive/"
os.chdir(new_dir)

# Get the current working directory
current_dir = os.getcwd()
print(f"Current working directory after changing: {current_dir}")

Mounted at /content/drive
Current working directory after changing: /content/drive/MyDrive


In [4]:
# ----------------------------
# Hardware check
# ----------------------------

def check_hardware() -> torch.device:
    if not torch.cuda.is_available():
        logger.warning("No GPU detected. Using CPU.")
        return torch.device("cpu")
    device_index = torch.cuda.current_device()
    device = torch.device(f"cuda:{device_index}")
    name = torch.cuda.get_device_name()
    mem = torch.cuda.get_device_properties().total_memory / 1024**3
    print(f"Training on GPU: {name} with {mem:.1f} GB")
    return device

import multiprocessing
# sklearn and multiprocessing
cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {cores}")
check_hardware()

Number of CPU cores: 8
Training on GPU: Tesla T4 with 14.7 GB


device(type='cuda', index=0)

***

Applies ReLU to focus on features positively influencing the class prediction

**.eval():**

- Action: This sets the model to "evaluation mode". This primarily affects two types of layers:

- Dropout Layers: They are deactivated. During training, dropout randomly sets some neuron activations to zero to prevent overfitting. During evaluation/inference, you want the full network's learned capacity, so dropout is turned off.

- Batch Normalization Layers: They switch from using the mean/variance of the current mini-batch to using their internally stored running estimates of mean/variance (accumulated during training). This provides stable normalization based on the overall training data distribution, which is desired for consistent evaluation.

- **Purpose**: To ensure the model behaves deterministically and uses its learned parameters in a way suitable for inference or analysis, rather than training.

- eval() mode itself does not disable gradient calculation; only operations within a with torch.no_grad(): context manager do that globally.

***

Core Idea of CAM Methods:
They aim to produce a heatmap overlayed on the input image, highlighting regions that strongly influenced the model's decision towards a particular class. They typically achieve this by combining feature maps from a late convolutional layer with some form of weights indicating the importance of each feature map for the target class.

***

In [5]:
#@title Setup

# --- Configuration ---
@dataclass
class Config:
    """Minimal config for model loading and analysis."""
    dropout_rate: float = 0.5
    data_dir: Path = Path("./batdrive/OC/P6/P6_data/Images/")
    data_csv: Path = Path("./batdrive/OC/P6/P6_data/df_cleaned.csv")
    output_dir: Path = Path("./batdrive/models/outputs/resnet/")
    required_columns: list = field(default_factory=lambda: ["image", "level_1"])

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)


# --- Data Loading ---
def validate_csv_columns(csv_path: Path, required_columns: list) -> None:
    """Check if the required columns exist in the CSV."""
    df_cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    missing = set(required_columns) - set(df_cols)
    if missing:
        raise ValueError(f"Missing required columns in CSV: {missing}")


def load_dataframe(csv_path: Path, required_columns: list) -> pd.DataFrame:
    """Validate columns and load the dataframe."""
    validate_csv_columns(csv_path, required_columns)
    return pd.read_csv(csv_path, usecols=required_columns)


# --- Model Definition ---
def setup_model(num_classes: int, dropout_rate: float) -> nn.Module:
    """Builds a ResNet34 model with a custom classification head."""
    model = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final fully-connected layer
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, num_classes)
    )
    return model

def get_transforms() -> A.Compose:
    return A.Compose([
        A.SmallestMaxSize(max_size=256, interpolation=cv2.INTER_AREA),
        A.CenterCrop(height=224, width=224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# --- INITIALIZATION ---
print("--- Running prerequisite setup ---")

# 1. Initialize configuration
cfg = Config()

# 2. Load the misclassified samples dataframe
try:
    misclassified_path = cfg.output_dir / "misclassified_samples.csv"
    df_mis = pd.read_csv(misclassified_path, index_col='Image File')
    print(f"✅ Loaded {len(df_mis)} misclassified samples from: {misclassified_path}")
except FileNotFoundError:
    print(f"❌ ERROR: Could not find misclassified samples file at {misclassified_path}")
    df_mis = None # Ensure df_mis exists to avoid crashing the next cell

print("--- Prerequisite setup complete ---")

--- Running prerequisite setup ---
✅ Loaded 41 misclassified samples from: batdrive/models/outputs/resnet/misclassified_samples.csv
--- Prerequisite setup complete ---


In [6]:
#@title device & data/model loading

# 1️⃣ Device setup
use_cuda = torch.cuda.is_available()
device   = torch.device("cuda" if use_cuda else "cpu")
print(f"Using device: {device}")

# 2️⃣ Load misclassified DataFrame
df_mis = df_mis.copy()
print(f"Loaded {len(df_mis)} misclassified samples.")

# 3️⃣ Get class names
df_all = load_dataframe(cfg.data_csv, cfg.required_columns)
le     = LabelEncoder().fit(df_all["level_1"])
class_names = le.classes_.tolist()
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

# 4️⃣ Build & load model
resnet_model = setup_model(num_classes, cfg.dropout_rate)
state_dict   = torch.load(cfg.output_dir / "best_phase2.pth", map_location="cpu")
resnet_model.load_state_dict(state_dict)
resnet_model.to(device).eval()

# 5️⃣ Make sure gradients are enabled
for p in resnet_model.parameters():
    p.requires_grad_(True)
print("Model ready for GradCAM.\n")


Using device: cuda
Loaded 41 misclassified samples.
Found 7 classes: ['Baby Care', 'Beauty and Personal Care', 'Computers', 'Home Decor & Festive Needs', 'Home Furnishing', 'Kitchen & Dining', 'Watches']
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 199MB/s]


Model ready for GradCAM.



In [7]:
# ==============================================================================
# PART 1: INITIALIZE GRAD-CAM FOR RESNET
# ==============================================================================
print("\n--- [1/2] Initializing Grad-CAM for ResNet ---")

cam = None

# This assumes 'resnet_model' is a loaded PyTorch model
if 'resnet_model' in locals() and resnet_model is not None:
    # Target the last convolutional block of the ResNet's fourth layer.
    # This is a common and effective target for ResNet architectures.
    target_layers = [resnet_model.layer4[-1]]

    # Instantiate GradCAM. No reshape_transform is needed for CNNs like ResNet.
    cam = GradCAM(model=resnet_model, target_layers=target_layers)
    print("✅ Grad-CAM is ready for ResNet.")
else:
    print("⚠️ Skipping Grad-CAM initialization: `resnet_model` not found.")

# This check ensures the misclassified dataframe is ready for the next step
if 'df_mis' not in locals() or df_mis.empty:
     print("⚠️ Misclassified DataFrame `df_mis` not found or is empty.")

# ==============================================================================
# PART 2: GENERATE & PLOT ALL MISCLASSIFIED EXPLANATIONS
# ==============================================================================
print("\n--- [2/2] Generating visualizations ---")
import matplotlib.pyplot as plt
import numpy as np
import cv2

def overlay_cam_on_image(img_rgb: np.ndarray, heatmap: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Resizes a heatmap to match the image and overlays it."""
    # Resize and apply colormap to the heatmap
    heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_MAGMA)
    heatmap_color_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    # Blend the heatmap with the original image
    overlayed_img = cv2.addWeighted(img_rgb, 1 - alpha, heatmap_color_rgb, alpha, 0)
    return overlayed_img

# This check ensures the previous cell ran successfully
if 'df_mis' in locals() and not df_mis.empty and cam is not None:

    total_samples = len(df_mis)
    print(f"\n--- Generating Grad-CAM explanations for all {total_samples} misclassified samples ---")

    # Loop directly over the DataFrame index to process every misclassified image
    for i, fname in enumerate(df_mis.index):
        print(f"Processing sample {i+1}/{total_samples}: {fname}")
        row = df_mis.loc[fname]
        true_lbl, pred_lbl = row["True class"], row["Predicted class"]
        t_i, p_i = class_names.index(true_lbl), class_names.index(pred_lbl)

        # Load and preprocess the image
        img_bgr = cv2.imread(str(cfg.data_dir / fname))
        if img_bgr is None:
            print(f"– ⚠️ couldn’t read {fname}, skipping")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        input_tensor = get_transforms()(image=img_rgb)["image"].unsqueeze(0).to(device)

        # Define targets for both the true and predicted classes
        targets_true = [ClassifierOutputTarget(t_i)]
        targets_pred = [ClassifierOutputTarget(p_i)]

        # Generate the CAMs using the Eigen-Smooth enhancement
        grayscale_cam_true = cam(input_tensor=input_tensor, targets=targets_true, eigen_smooth=True)[0, :]
        grayscale_cam_pred = cam(input_tensor=input_tensor, targets=targets_pred, eigen_smooth=True)[0, :]

        # Create overlay visualizations
        vis_true = overlay_cam_on_image(img_rgb, grayscale_cam_true)
        vis_pred = overlay_cam_on_image(img_rgb, grayscale_cam_pred)

        # Plot the results: Original, CAM for True Class, CAM for Predicted Class
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f"File: {fname}", fontsize=14)
        axes[0].imshow(img_rgb)
        axes[0].set_title("Original")
        axes[0].axis("off")

        axes[1].imshow(vis_true)
        axes[1].set_title(f"CAM for True: {true_lbl}")
        axes[1].axis("off")

        axes[2].imshow(vis_pred)
        axes[2].set_title(f"CAM for Pred: {pred_lbl}")
        axes[2].axis("off")

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    print(f"--- Done plotting all {total_samples} samples ---")
else:
    print("⚠️ Plotting skipped due to setup errors or empty misclassified list.")

Output hidden; open in https://colab.research.google.com to view.

In [8]:
#@title cleanup

# pytorch-grad-cam handles hooks automatically,
# but you can delete the object if you’re done:
del cam
print("Cleaned up GradCAM object.")

Cleaned up GradCAM object.


***